# Scalar Chebyshev2 vs Trapezoid Integration

This notebook reproduces the scalar noisy integration comparison on `[0, 1]`. The Chebyshev path is implemented in the reusable library code and calls `gtsam.Chebyshev2` directly for the pseudo-spectral fit and quadrature weights. The trapezoid baseline is local NumPy code because this is not provided by GTSAM.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve()
if not (repo_root / "python").exists():
    repo_root = repo_root.parent
python_dir = repo_root / "python"
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))

import imuFactors.scalar_quadrature as scalar_quadrature

plt.rcParams.update({"figure.dpi": 120})

In [ ]:
INTERVAL = (0.0, 1.0)
FUNCTIONS = [
    scalar_quadrature.ScalarFunction(
        name="tanh(t)",
        value=lambda t: np.tanh(t),
        antiderivative=lambda t: np.log(np.cosh(t)),
    ),
    scalar_quadrature.ScalarFunction(
        name="sin(2*pi*t)",
        value=lambda t: np.sin(2.0 * np.pi * t),
        antiderivative=lambda t: (1.0 - np.cos(2.0 * np.pi * t)) / (2.0 * np.pi),
    ),
    scalar_quadrature.ScalarFunction(
        name="pi*t^(31/4)",
        value=lambda t: np.pi * t ** (31.0 / 4.0),
        antiderivative=lambda t: np.pi * t ** (35.0 / 4.0) / (35.0 / 4.0),
    ),
]

SAMPLE_COUNTS = np.arange(5, 49)
CHEBYSHEV_NODES = np.arange(2, 25)
FIXED_NODES = [2, 9, 17, 24]
NOISE_FRACTIONS = np.array([
    0.0, 0.025, 0.05, 0.06, 0.075, 0.10,
    0.12, 0.15, 0.17, 0.20, 0.225,
])
SELECTED_NOISE_FRACTIONS = [0.06, 0.12, 0.17, 0.225]
NUM_SEEDS = 100
RANDOM_SEED = 20260523
EVALUATION_COUNT = 151

In [ ]:
result = scalar_quadrature.run_scalar_monte_carlo(
    FUNCTIONS,
    sample_counts=SAMPLE_COUNTS,
    chebyshev_node_counts=CHEBYSHEV_NODES,
    noise_fractions=NOISE_FRACTIONS,
    num_seeds=NUM_SEEDS,
    seed=RANDOM_SEED,
    interval=INTERVAL,
    evaluation_count=EVALUATION_COUNT,
)

method_metrics = result.method_metrics
comparisons = result.comparisons
comparisons.head()

In [ ]:
fig = scalar_quadrature.plot_fixed_node_comparison(
    comparisons,
    function_name="tanh(t)",
    selected_nodes=FIXED_NODES,
)
fig

In [ ]:
fig = scalar_quadrature.plot_function_noise_comparison(
    comparisons,
    function_names=[function.name for function in FUNCTIONS],
    selected_noise_fractions=SELECTED_NOISE_FRACTIONS,
    metric="end_error",
)
fig

In [ ]:
summary = (
    comparisons.groupby("function")[["end_error", "rmse_error", "max_error"]]
    .agg(["min", "median", "max"])
)
summary